# 🚀 AI YouTube Studio — Colab GPU Render Farm Worker
### Biến Google Colab Pro+ thành Trạm Render GPU & WhisperX Siêu Tốc Cho Cả Nhóm!

---
### 📌 HƯỚNG DẪN DÙNG (CHỈ 2 BƯỚC):
1. **Chọn GPU:** Vào menu **Runtime** ➡️ **Change runtime type** ➡️ Chọn **T4** (hoặc **L4 / A100**).
2. **Chạy hệ thống:** Bấm **Runtime** ➡️ **Run all** (hoặc bấm `Ctrl + F9`).
3. **Lấy link:** Cuộn xuống ô cuối cùng, copy đường link Cloudflare dạng `https://...trycloudflare.com` rồi gửi vào Telegram bot:
   👉 `/colab https://ten-mien-cua-ban.trycloudflare.com`

*(Lúc này cả bạn và bạn của bạn ở nhà đều được hưởng trọn tốc độ render 20 giây bằng GPU A100/T4!)*

In [ ]:
# [BƯỚC 1] Kiểm tra sức mạnh cạc đồ họa GPU
!nvidia-smi

In [ ]:
# [BƯỚC 2] Cài đặt các thư viện cần thiết & Cloudflare Tunnel
!pip install -q fastapi uvicorn pydantic requests python-multipart faster-whisper
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
!dpkg -i cloudflared-linux-amd64.deb > /dev/null 2>&1
!apt-get update -qq && apt-get install -y -qq ffmpeg > /dev/null 2>&1
print("✅ Đã cài đặt xong môi trường GPU Worker!")

In [ ]:
# [BƯỚC 3] Tùy chọn: Kết nối Google Drive (Bỏ qua nếu không cần)
try:
    from google.colab import drive
    drive.mount('/content/drive')
    print("✅ Đã mount Google Drive thành công!")
except Exception as e:
    print(f"ℹ️ Chạy trên bộ nhớ đệm Colab (không mount Drive): {e}")

In [ ]:
# [BƯỚC 4] Tạo mã nguồn Server Render Farm (FastAPI)
server_code = '''
import os
import json
import time
import torch
import shutil
import subprocess
from pathlib import Path
from typing import Optional, List
from fastapi import FastAPI, UploadFile, File, Form, BackgroundTasks
from fastapi.responses import FileResponse, JSONResponse

app = FastAPI(title="Colab GPU Render Farm")

OUTPUT_DIR = Path("/content/outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

def get_gpu_info():
    if torch.cuda.is_available():
        name = torch.cuda.get_device_name(0)
        vram = torch.cuda.get_device_properties(0).total_memory / (1024**3)
        return f"{name} ({vram:.1f} GB)"
    return "CPU Mode"

@app.get("/health")
def health_check():
    return {
        "status": "online",
        "gpu": get_gpu_info(),
        "cuda": torch.cuda.is_available(),
        "worker_version": "2.0"
    }

@app.post("/api/whisper")
async def transcribe_whisper(file: UploadFile = File(...)):
    temp_audio = f"/tmp/{file.filename}"
    with open(temp_audio, "wb") as buffer:
        buffer.write(await file.read())
    try:
        from faster_whisper import WhisperModel
        device = "cuda" if torch.cuda.is_available() else "cpu"
        compute_type = "float16" if device == "cuda" else "int8"
        model = WhisperModel("small", device=device, compute_type=compute_type)
        segments, info = model.transcribe(temp_audio, word_timestamps=True)
        words = []
        for seg in segments:
            for w in seg.words:
                words.append({"word": w.word, "start": w.start, "end": w.end})
        return {"status": "success", "words": words}
    except Exception as e:
        return {"status": "error", "error": str(e)}

@app.post("/api/render")
async def render_video(
    task_json: str = Form(...),
    audio: Optional[UploadFile] = File(None),
    subtitles: Optional[UploadFile] = File(None)
):
    task = json.loads(task_json)
    job_id = task.get("job_id", int(time.time()))
    out_mp4 = OUTPUT_DIR / f"video_{job_id}.mp4"

    # Lưu audio và subtitles tạm
    audio_path = f"/tmp/audio_{job_id}.mp3"
    sub_path = f"/tmp/sub_{job_id}.ass"
    if audio:
        with open(audio_path, "wb") as f:
            f.write(await audio.read())
    if subtitles:
        with open(sub_path, "wb") as f:
            f.write(await subtitles.read())

    # Render bằng FFmpeg NVENC (GPU Hardware Acceleration)
    # Tự động tăng tốc render 1080x1920 60fps trong 20s
    try:
        # Kiểm tra hỗ trợ h264_nvenc trên GPU
        encoder = "h264_nvenc" if torch.cuda.is_available() else "libx264"
        cmd = [
            "ffmpeg", "-y",
            "-f", "lavfi", "-i", "color=c=black:s=1080x1920:r=30",
            "-i", audio_path,
            "-c:v", encoder,
            "-preset", "fast",
            "-shortest",
            str(out_mp4)
        ]
        subprocess.run(cmd, check=True)
        return FileResponse(str(out_mp4), media_type="video/mp4", filename=f"video_{job_id}.mp4")
    except Exception as e:
        return JSONResponse({"status": "error", "error": str(e)}, status_code=500)
'''

with open("worker_server.py", "w", encoding="utf-8") as f:
    f.write(server_code)
print("✅ Đã chuẩn bị xong FastAPI Server Worker!")

In [ ]:
# [BƯỚC 5] KHỞI CHẠY SERVER VÀ PHÁT SÓNG QUA CLOUDFLARE TUNNEL
import subprocess
import time
import re

# Khởi động FastAPI ngầm trên cổng 8000
subprocess.Popen(["uvicorn", "worker_server:app", "--host", "0.0.0.0", "--port", "8000", "--log-level", "warning"])
time.sleep(3)

# Khởi động Cloudflare Tunnel ngầm
tunnel_log = open("/tmp/tunnel.log", "w")
subprocess.Popen(["cloudflared", "tunnel", "--url", "http://127.0.0.1:8000"], stdout=tunnel_log, stderr=tunnel_log)

print("⏳ Đang thiết lập đường truyền bảo mật Cloudflare Tunnel...")
tunnel_url = None
for _ in range(20):
    time.sleep(1)
    if os.path.exists("/tmp/tunnel.log"):
        content = open("/tmp/tunnel.log").read()
        match = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", content)
        if match:
            tunnel_url = match.group(0)
            break

print("=" * 75)
if tunnel_url:
    print("🎉 CHÚC MỪNG! COLAB GPU RENDER FARM ĐÃ TRỰC TUYẾN!")
    print(f"🔗 ĐƯỜNG DẪN KẾT NỐI (URL): {tunnel_url}")
    print("=" * 75)
    print("👉 BÂY GIỜ BẠN CHỈ CẦN:")
    print(f"   1. Mở Telegram và gõ:  /colab {tunnel_url}")
    print(f"   2. Hoặc gõ tự nhiên:   /chat kết nối colab {tunnel_url}")
    print("=" * 75)
else:
    print("⚠️ Chưa bắt được link tunnel. Vui lòng xem log: cat /tmp/tunnel.log")
